<a href="https://colab.research.google.com/github/Karthikreddy1010/Electric-poles-and-wires-detection/blob/main/Electric_Wires_labels_Checking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install opencv-python-headless shapely

In [ ]:
!unzip /content/Wire_Labeling_Ideal.v2-wire_set_polygons_100.yolov8.zip

In [4]:
import argparse
import os
import random
import shutil
import sys # Import sys

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}


def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--root", default="/content", help="Folder containing train/images and train/labels")
    parser.add_argument("--val-frac", type=float, default=0.2, help="Fraction of images to move into valid/ (default 0.2)")
    parser.add_argument("--seed", type=int, default=42, help="Random seed for reproducible split")
    parser.add_argument("--dry-run", action="store_true", help="Print what would happen without moving any files")

    # Preserve original sys.argv
    original_argv = sys.argv
    # Set sys.argv to only the script name to avoid issues with kernel-injected arguments
    sys.argv = [original_argv[0]]
    # Parse only known arguments
    args, unknown = parser.parse_known_args()
    # Restore original sys.argv
    sys.argv = original_argv

    train_images_dir = os.path.join(args.root, "train", "images")
    train_labels_dir = os.path.join(args.root, "train", "labels")
    valid_images_dir = os.path.join(args.root, "valid", "images")
    valid_labels_dir = os.path.join(args.root, "valid", "labels")

    if not os.path.isdir(train_images_dir):
        raise SystemExit(f"ERROR: {train_images_dir} not found. Check --root.")
    if not os.path.isdir(train_labels_dir):
        raise SystemExit(f"ERROR: {train_labels_dir} not found. Check --root.")

    stems = []
    for fname in sorted(os.listdir(train_images_dir)):
        stem, ext = os.path.splitext(fname)
        if ext.lower() in IMG_EXTS:
            stems.append((stem, fname))

    total = len(stems)
    if total == 0:
        raise SystemExit(f"ERROR: no images found in {train_images_dir}")

    n_val = max(1, round(total * args.val_frac))
    print(f"Found {total} images in train/. Moving {n_val} ({args.val_frac*100:.0f}%) into valid/.")

    random.seed(args.seed)
    shuffled = stems[:]
    random.shuffle(shuffled)
    val_set = shuffled[:n_val]

    if not args.dry_run:
        os.makedirs(valid_images_dir, exist_ok=True)
        os.makedirs(valid_labels_dir, exist_ok=True)

    moved = 0
    missing_labels = []
    for stem, img_fname in val_set:
        label_fname = stem + ".txt"
        src_img = os.path.join(train_images_dir, img_fname)
        src_label = os.path.join(train_labels_dir, label_fname)
        dst_img = os.path.join(valid_images_dir, img_fname)
        dst_label = os.path.join(valid_labels_dir, label_fname)

        if not os.path.exists(src_label):
            missing_labels.append(img_fname)

        if args.dry_run:
            print(f"  WOULD MOVE: {img_fname}" + (" (+label)" if os.path.exists(src_label) else " (NO LABEL FOUND)"))
            continue

        shutil.move(src_img, dst_img)
        if os.path.exists(src_label):
            shutil.move(src_label, dst_label)
        moved += 1

    remaining = total - moved if not args.dry_run else total - len(val_set)

    print("\n--- Done ---" if not args.dry_run else "\n--- Dry run complete, nothing moved ---")
    print(f"train/: {remaining} images remaining")
    print(f"valid/: {moved if not args.dry_run else len(val_set)} images")
    if missing_labels:
        print(f"\nWARNING: {len(missing_labels)} image(s) moved with no matching label file: {missing_labels[:5]}{'...' if len(missing_labels) > 5 else ''}")

    print("\nNext step: update data.yaml so the val: line points to the new valid/images folder, e.g.")
    print("  val: ../valid/images")
    print("(relative to the folder data.yaml sits in), then re-run the dataset validator on both splits:")
    print(f"  !python validate_yolo_seg_dataset.py --root {args.root} --yaml {args.root}/data.yaml")


if __name__ == "__main__":
    main()

Found 100 images in train/. Moving 20 (20%) into valid/.

--- Done ---
train/: 80 images remaining
valid/: 20 images

Next step: update data.yaml so the val: line points to the new valid/images folder, e.g.
  val: ../valid/images
(relative to the folder data.yaml sits in), then re-run the dataset validator on both splits:
  !python validate_yolo_seg_dataset.py --root /content --yaml /content/data.yaml


In [6]:
import argparse
import csv
import os
import sys
from collections import defaultdict

import cv2
import numpy as np

try:
    from shapely.geometry import Polygon
    from shapely.validation import explain_validity
    HAVE_SHAPELY = True
except ImportError:
    HAVE_SHAPELY = False

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

# Small-object thresholds (tune for your use case). These are percentages
# of total image area. Wires/cables are expected to be small, but anything
# below MIN_AREA_PCT_WARN is worth knowing about because YOLO's downsampling
# strides (8/16/32) can make these vanish from deeper feature maps.
MIN_AREA_PCT_WARN = 0.05   # warn below this
MIN_AREA_PCT_FAIL = 0.005  # flag as near-degenerate below this


def load_class_names(yaml_path):
    """Minimal YAML parser for the 'names' field, no external dependency required."""
    if not yaml_path or not os.path.exists(yaml_path):
        return {}
    names = {}
    with open(yaml_path, "r") as f:
        text = f.read()
    # Try a quick literal-list parse: names: ['a', 'b', 'c']
    import re
    m = re.search(r"names:\s*\[(.*?)\]", text, re.S)
    if m:
        items = [x.strip().strip("'\"") for x in m.group(1).split(",")]
        for i, name in enumerate(items):
            names[i] = name
        return names
    # Fallback: names:\n  0: foo\n  1: bar
    m2 = re.search(r"names:\s*\n((?:\s+\d+:\s*.+\n?)+)", text)
    if m2:
        for line in m2.group(1).splitlines():
            line = line.strip()
            if not line:
                continue
            k, v = line.split(":", 1)
            names[int(k.strip())] = v.strip().strip("'\"")
    return names


def find_pairs(images_dir, labels_dir):
    """Return list of (image_path, label_path_or_None) and orphan label files."""
    pairs = []
    img_stems = {}
    if os.path.isdir(images_dir):
        for fname in sorted(os.listdir(images_dir)):
            stem, ext = os.path.splitext(fname)
            if ext.lower() in IMG_EXTS:
                img_stems[stem] = os.path.join(images_dir, fname)

    label_stems = set()
    if os.path.isdir(labels_dir):
        for fname in sorted(os.listdir(labels_dir)):
            if fname.lower().endswith(".txt"):
                label_stems.add(os.path.splitext(fname)[0])

    for stem, img_path in img_stems.items():
        label_path = os.path.join(labels_dir, stem + ".txt")
        pairs.append((img_path, label_path if stem in label_stems else None))

    orphan_labels = [
        os.path.join(labels_dir, s + ".txt")
        for s in label_stems
        if s not in img_stems
    ]
    return pairs, orphan_labels


def parse_label_file(label_path, n_classes):
    """Parse a YOLO-seg label file. Returns (instances, issues).

    Each instance: dict with keys class_id, points (Nx2 normalized), raw_line, line_no
    Each issue: dict with keys line_no, severity, message
    """
    instances = []
    issues = []
    if not os.path.exists(label_path):
        return instances, issues

    with open(label_path, "r") as f:
        raw_lines = [l.rstrip("\n") for l in f]

    for line_no, line in enumerate(raw_lines, start=1):
        stripped = line.strip()
        if not stripped:
            continue
        parts = stripped.split()

        # class id check
        try:
            cls = int(parts[0])
        except (ValueError, IndexError):
            issues.append({"line_no": line_no, "severity": "ERROR",
                            "message": f"Could not parse class id from: '{line}'"})
            continue

        if cls < 0 or (n_classes and cls >= n_classes):
            issues.append({"line_no": line_no, "severity": "ERROR",
                            "message": f"Class id {cls} outside expected range [0,{n_classes-1}]"})

        coord_strs = parts[1:]

        # detect bounding-box-style line (exactly 4 numeric fields after class)
        # which would mean this is NOT a segmentation file
        if len(coord_strs) == 4:
            issues.append({"line_no": line_no, "severity": "WARNING",
                            "message": "Only 4 coordinate values found - looks like a "
                                       "bounding-box (detection) annotation, not a polygon. "
                                       "Mixed-format files will break a segmentation dataloader."})

        if len(coord_strs) % 2 != 0:
            issues.append({"line_no": line_no, "severity": "ERROR",
                            "message": f"Odd number of coordinate values ({len(coord_strs)}); "
                                       f"cannot form x,y pairs."})
            continue

        try:
            vals = [float(x) for x in coord_strs]
        except ValueError:
            issues.append({"line_no": line_no, "severity": "ERROR",
                            "message": "Non-numeric coordinate value found."})
            continue

        out_of_range = [v for v in vals if v < 0.0 or v > 1.0]
        if out_of_range:
            issues.append({"line_no": line_no, "severity": "ERROR",
                            "message": f"{len(out_of_range)} coordinate(s) outside [0,1] range "
                                       f"(min={min(vals):.4f}, max={max(vals):.4f})"})

        points = [(vals[i], vals[i + 1]) for i in range(0, len(vals), 2)]

        if len(points) < 3:
            issues.append({"line_no": line_no, "severity": "ERROR",
                            "message": f"Polygon has only {len(points)} point(s); "
                                       f"needs >= 3 to form a shape."})
            continue

        instances.append({
            "class_id": cls,
            "points": points,
            "line_no": line_no,
        })

    return instances, issues


def analyze_instance(inst, img_w, img_h):
    """Compute geometric stats for one polygon instance. Returns dict of stats + issues list."""
    issues = []
    pts_px = np.array([[x * img_w, y * img_h] for x, y in inst["points"]], dtype=np.float32)

    area_px = float(cv2.contourArea(pts_px))
    area_pct = area_px / (img_w * img_h) * 100.0

    is_valid_geom = True
    validity_reason = ""
    if HAVE_SHAPELY:
        try:
            poly = Polygon(inst["points"])
            is_valid_geom = poly.is_valid
            if not is_valid_geom:
                validity_reason = explain_validity(poly)
        except Exception as e:
            is_valid_geom = False
            validity_reason = str(e)

    if not is_valid_geom:
        issues.append({"line_no": inst["line_no"], "severity": "ERROR",
                        "message": f"Self-intersecting / invalid polygon geometry: {validity_reason}"})

    if area_pct < MIN_AREA_PCT_FAIL:
        issues.append({"line_no": inst["line_no"], "severity": "WARNING",
                        "message": f"Near-degenerate polygon area ({area_pct:.4f}% of image, "
                                   f"{area_px:.1f}px) - likely collapsed/duplicate points or a "
                                   f"mis-click rather than a real wire segment."})
    elif area_pct < MIN_AREA_PCT_WARN:
        issues.append({"line_no": inst["line_no"], "severity": "INFO",
                        "message": f"Very small object ({area_pct:.4f}% of image) - high risk of "
                                   f"being lost at deeper YOLO feature map strides (16/32)."})

    return {
        "class_id": inst["class_id"],
        "n_points": len(inst["points"]),
        "area_px": area_px,
        "area_pct": area_pct,
        "is_valid_geom": is_valid_geom,
    }, issues


def draw_overlay(img, instances, class_names, out_path):
    h, w = img.shape[:2]
    palette = [(0, 0, 255), (0, 255, 0), (255, 0, 0), (0, 255, 255),
               (255, 0, 255), (255, 255, 0), (128, 0, 255), (0, 128, 255)]
    overlay = img.copy()
    for inst in instances:
        cls = inst["class_id"]
        color = palette[cls % len(palette)]
        pts = np.array([[x * w, y * h] for x, y in inst["points"]], dtype=np.int32)
        filled = overlay.copy()
        cv2.fillPoly(filled, [pts], color)
        overlay = cv2.addWeighted(filled, 0.25, overlay, 0.75, 0)
        cv2.polylines(overlay, [pts], isClosed=True, color=color, thickness=2)
    cv2.imwrite(out_path, overlay)


def run(args):
    class_names = load_class_names(args.yaml)
    n_classes = len(class_names) if class_names else 0
    if class_names:
        print(f"Loaded {n_classes} class names from data.yaml: {class_names}")
    else:
        print("WARNING: could not load class names from data.yaml; class-id range "
              "checks will be skipped.")

    if args.images_dir and args.labels_dir:
        split_dirs = {"default": (args.images_dir, args.labels_dir)}
    else:
        split_dirs = {}
        for split in args.splits:
            images_dir = os.path.join(args.root, split, "images")
            labels_dir = os.path.join(args.root, split, "labels")
            if os.path.isdir(images_dir):
                split_dirs[split] = (images_dir, labels_dir)
            else:
                print(f"NOTE: split '{split}' not found at {images_dir}, skipping.")

    if not split_dirs:
        print("ERROR: no image directories found. Check --root/--splits or pass "
              "--images-dir/--labels-dir directly.")
        sys.exit(1)

    all_instance_rows = []
    all_issue_rows = []
    class_counts = defaultdict(lambda: defaultdict(int))  # split -> class_id -> count
    image_sizes = defaultdict(set)  # split -> set of (w,h)
    missing_label_files = defaultdict(list)
    empty_label_files = defaultdict(list)
    total_images = defaultdict(int)

    overlay_dir = os.path.join(args.out_dir, "overlays")
    if args.overlay_samples > 0:
        os.makedirs(overlay_dir, exist_ok=True)

    for split, (images_dir, labels_dir) in split_dirs.items():
        pairs, orphan_labels = find_pairs(images_dir, labels_dir)
        total_images[split] = len(pairs)

        for orphan in orphan_labels:
            all_issue_rows.append({"split": split, "file": orphan, "line_no": "",
                                    "severity": "WARNING",
                                    "message": "Label file exists with no matching image."})

        overlays_done = 0
        for img_path, label_path in pairs:
            img_name = os.path.basename(img_path)

            if label_path is None:
                missing_label_files[split].append(img_name)
                all_issue_rows.append({"split": split, "file": img_name, "line_no": "",
                                        "severity": "ERROR",
                                        "message": "No matching label file found for this image."})
                continue

            img = cv2.imread(img_path)
            if img is None:
                all_issue_rows.append({"split": split, "file": img_name, "line_no": "",
                                        "severity": "ERROR",
                                        "message": "Could not read image file (corrupt or unsupported)."})
                continue
            h, w = img.shape[:2]
            image_sizes[split].add((w, h))

            instances, parse_issues = parse_label_file(label_path, n_classes)
            for iss in parse_issues:
                all_issue_rows.append({"split": split, "file": os.path.basename(label_path),
                                        **iss})

            if not instances and not parse_issues:
                empty_label_files[split].append(img_name)
                all_issue_rows.append({"split": split, "file": os.path.basename(label_path),
                                        "line_no": "", "severity": "WARNING",
                                        "message": "Label file is empty (no annotations) - "
                                                   "image will be treated as background/negative."})

            for inst in instances:
                stats, geom_issues = analyze_instance(inst, w, h)
                class_counts[split][inst["class_id"]] += 1
                all_instance_rows.append({
                    "split": split,
                    "file": img_name,
                    "line_no": inst["line_no"],
                    "class_id": inst["class_id"],
                    "class_name": class_names.get(inst["class_id"], str(inst["class_id"])),
                    "n_points": stats["n_points"],
                    "area_px": round(stats["area_px"], 2),
                    "area_pct_of_image": round(stats["area_pct"], 4),
                    "img_w": w,
                    "img_h": h,
                    "is_valid_geometry": stats["is_valid_geom"],
                })
                for iss in geom_issues:
                    all_issue_rows.append({"split": split, "file": os.path.basename(label_path),
                                            **iss})

            if args.overlay_samples > 0 and overlays_done < args.overlay_samples and instances:
                out_path = os.path.join(overlay_dir, f"{split}_{os.path.splitext(img_name)[0]}_overlay.png")
                draw_overlay(img, instances, class_names, out_path)
                overlays_done += 1

    # ---- write CSVs ----
    os.makedirs(args.out_dir, exist_ok=True)
    inst_csv = os.path.join(args.out_dir, "report_instances.csv")
    issues_csv = os.path.join(args.out_dir, "report_issues.csv")

    if all_instance_rows:
        with open(inst_csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=list(all_instance_rows[0].keys()))
            writer.writeheader()
            writer.writerows(all_instance_rows)

    if all_issue_rows:
        fieldnames = sorted(set(k for row in all_issue_rows for k in row.keys()))
        with open(issues_csv, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(all_issue_rows)

    # ---- print summary report ----
    print("\n" + "=" * 70)
    print("YOLO SEGMENTATION DATASET VALIDATION REPORT")
    print("=" * 70)

    grand_total_images = sum(total_images.values())
    print(f"\nTotal images found across splits: {grand_total_images}")
    for split in split_dirs:
        print(f"  {split:8s}: {total_images[split]} images")

    if grand_total_images < 300:
        print(f"\n  >>> DATASET SIZE WARNING: {grand_total_images} images is small for training a "
              f"segmentation model from scratch, especially for thin/low-area objects like "
              f"wires. Expect high variance between training runs and a real risk of "
              f"overfitting. Strongly consider: starting from a pretrained YOLO-seg checkpoint "
              f"(transfer learning, not training from scratch), heavy augmentation, and/or "
              f"treating current results as a feasibility pilot rather than a final model. "
              f"More data will move the needle more than further hyperparameter tuning at this size.")

    print("\n--- Image size consistency ---")
    for split, sizes in image_sizes.items():
        if len(sizes) == 1:
            print(f"  {split:8s}: consistent size {next(iter(sizes))}")
        else:
            print(f"  {split:8s}: INCONSISTENT sizes found: {sizes}")

    print("\n--- Missing / empty labels ---")
    any_missing = False
    for split in split_dirs:
        miss = missing_label_files.get(split, [])
        empty = empty_label_files.get(split, [])
        if miss or empty:
            any_missing = True
        print(f"  {split:8s}: {len(miss)} images with NO label file, {len(empty)} images with EMPTY label file")
    if not any_missing:
        print("  None - every image has a non-empty label file.")

    print("\n--- Class distribution (instance counts) ---")
    header = "  split    " + "  ".join(f"{class_names.get(c, c):>10}" for c in sorted(class_names) or sorted({c for d in class_counts.values() for c in d}))
    print(header)
    all_class_ids = sorted(class_names.keys()) if class_names else sorted({c for d in class_counts.values() for c in d})
    for split in split_dirs:
        row = f"  {split:8s}"
        for c in all_class_ids:
            row += f"  {class_counts[split].get(c, 0):>10}"
        print(row)

    totals_by_class = defaultdict(int)
    for split_d in class_counts.values():
        for c, n in split_d.items():
            totals_by_class[c] += n
    print("  " + "-" * 60)
    row = f"  {'TOTAL':8s}"
    for c in all_class_ids:
        row += f"  {totals_by_class.get(c, 0):>10}"
    print(row)

    for c in all_class_ids:
        if totals_by_class.get(c, 0) < 50:
            print(f"  >>> CLASS IMBALANCE / SCARCITY WARNING: class "
                  f"'{class_names.get(c, c)}' has only {totals_by_class.get(c,0)} total instances. "
                  f"Models tend to underperform badly on classes this rare; consider merging "
                  f"classes, collecting more examples, or using class-weighted loss.")

    print("\n--- Object size distribution (area as % of image) ---")
    if all_instance_rows:
        areas = [r["area_pct_of_image"] for r in all_instance_rows]
        areas_sorted = sorted(areas)
        n = len(areas_sorted)
        p50 = areas_sorted[n // 2]
        p10 = areas_sorted[int(n * 0.1)]
        n_small_warn = sum(1 for a in areas if a < MIN_AREA_PCT_WARN)
        n_small_fail = sum(1 for a in areas if a < MIN_AREA_PCT_FAIL)
        print(f"  Total polygon instances analyzed: {n}")
        print(f"  Median area: {p50:.4f}% of image | 10th percentile: {p10:.4f}% of image")
        print(f"  Instances below {MIN_AREA_PCT_WARN}% (small-object risk): {n_small_warn} "
              f"({n_small_warn/n*100:.1f}%)")
        print(f"  Instances below {MIN_AREA_PCT_FAIL}% (near-degenerate, check manually): {n_small_fail}")
        if n_small_warn / n > 0.3:
            print(f"  >>> Over 30% of your instances are very small. At YOLO's default imgsz=640, "
                  f"many of these will be lost in deeper feature maps. Strongly consider training "
                  f"at a higher --imgz (960-1280) and reviewing P2/small-object head configs "
                  f"if using a custom YOLO architecture.")
    else:
        print("  No valid instances parsed.")

    print("\n--- Geometry / parsing issues ---")
    n_errors = sum(1 for r in all_issue_rows if r.get("severity") == "ERROR")
    n_warnings = sum(1 for r in all_issue_rows if r.get("severity") == "WARNING")
    n_info = sum(1 for r in all_issue_rows if r.get("severity") == "INFO")
    print(f"  ERROR:   {n_errors}  (will likely break training or silently corrupt masks)")
    print(f"  WARNING: {n_warnings}  (won't crash training, but worth reviewing)")
    print(f"  INFO:    {n_info}  (small-object notices, informational)")
    if n_errors:
        print(f"  See {issues_csv} for full details, filtered to severity=ERROR first.")

    if not HAVE_SHAPELY:
        print("\n  NOTE: shapely not installed - self-intersection checks were skipped. "
              "Install with `pip install shapely` for full geometry validation.")

    print(f"\nFull per-instance stats written to: {inst_csv}")
    print(f"Full issue list written to:         {issues_csv}")
    if args.overlay_samples > 0:
        print(f"Sample overlay images written to:   {overlay_dir}")
    print("=" * 70 + "\n")


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--root", default=".", help="Dataset root containing split folders (default: current dir)")
    parser.add_argument("--yaml", default=None, help="Path to data.yaml (for class names)")
    parser.add_argument("--splits", nargs="+", default=["train", "valid", "test"],
                         help="Split subfolder names to look for under --root")
    parser.add_argument("--images-dir", default=None, help="Directly specify a single images folder (overrides --root/--splits)")
    parser.add_argument("--labels-dir", default=None, help="Directly specify a single labels folder (overrides --root/--splits)")
    parser.add_argument("--out-dir", default="./validation_report", help="Where to write CSV reports and overlays")
    parser.add_argument("--overlay-samples", type=int, default=10, help="Number of overlay images to generate per split (0 to disable)")

    # Store original sys.argv
    original_argv = sys.argv
    # Temporarily set sys.argv to only the script name to avoid issues with kernel-injected arguments
    sys.argv = [original_argv[0]]
    # Parse only known arguments, allowing unknown ones to be ignored
    args, unknown = parser.parse_known_args()
    # Restore original sys.argv
    sys.argv = original_argv

    run(args)

NOTE: split 'test' not found at ./test/images, skipping.

YOLO SEGMENTATION DATASET VALIDATION REPORT

Total images found across splits: 100
  train   : 80 images
  valid   : 20 images

  >>> DATASET SIZE WARNING: 100 images is small for training a segmentation model from scratch, especially for thin/low-area objects like wires. Expect high variance between training runs and a real risk of overfitting. Strongly consider: starting from a pretrained YOLO-seg checkpoint (transfer learning, not training from scratch), heavy augmentation, and/or treating current results as a feasibility pilot rather than a final model. More data will move the needle more than further hyperparameter tuning at this size.

--- Image size consistency ---
  train   : consistent size (432, 432)
  valid   : consistent size (432, 432)

--- Missing / empty labels ---
  train   : 0 images with NO label file, 0 images with EMPTY label file
  valid   : 0 images with NO label file, 0 images with EMPTY label file
  None 

In [7]:
import shutil
import os
from google.colab import files

# Define the directory to be zipped and the output zip file name
directory_to_zip = "/content/validation_report"
output_zip_name = "validation_report.zip"

# Create the zip archive
shutil.make_archive(os.path.splitext(output_zip_name)[0], 'zip', directory_to_zip)

print(f"Folder '{directory_to_zip}' has been successfully zipped to '{output_zip_name}'.")

# Offer the file for download
files.download(output_zip_name)

Folder '/content/validation_report' has been successfully zipped to 'validation_report.zip'.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>